# CRS data acquisition overview 
This notebook outlines the functionality of the CRS instrument class and the measurement procedures. First, use the following instructions to ensure that the daq computer is set up properly and that the CRS firmware is up to date.
### Some definitions
<ul>
    <li> <b> module: </b> CRS readout module, corresponding to one ADC/DAC pair </li>
    <li> <b> channel: </b> I prefer "tone", but t0 uses "channel". Refers to one frequency/amplitude/phase output of a module. </li>
    <li> <b> ch_map: </b> This parameter was hidden within functions in the previous version of citkid. It is a dictionary that maps modules index to channel index, which is required to keep the upper level arrays as 1D while the internal code splits between modules. Now, each function that separates data between modules takes <i> ch_map </i> as an optional parameter, and will create it if it is not set. For procedures, it is recommended to create <i> ch_map </i> only once, to avoid channels jumping between modules, and potentially having power calibration issues. The user can specify whether they want missing channels (too far from any NCO) to raise an error, or result in NANs in the output data.
    </li>
</ul>

## Module installation 
Version 1.3.2 of the `rfmux` module should be installed via PyPI. Creating an environment via the [citkid ReadMe](https://github.com/loganfoote/citkid/) is highly encouraged, as it will ensure module version dependencies are correct. 

## Firmware setup 
Follow the instructions in the [rfmux Firmware ReadME](https://github.com/t0/rfmux/blob/main/docs/guides/firmware.md) to image the CRS SD card with firmware version 1.6.0rc3.

## Network setup 
Follow the steps in the [rfmux Networking ReadME](https://github.com/t0/rfmux/blob/main/docs/guides/networking.md). Also, make sure "Use this connection only for resources on its network" is selected in the IPv4 settings for the ethernet port. 

## System testing 
When configuring a new DAQ computer, make sure all `citkid` functions pass to ensure environment compatibility. Navigate to the repository directory and run 
```
pytest tests/crs
```
Ensure that all tests pass. Note that work-in-progress on the development branch may cause some tests to fail, so be aware of changes that are in progress. 

## Hardware testing
To test CRS hardware, set up the system in loopback mode and run 
```
pytest tests/crs/test_instrument_hardware.py --crs_sn=<sn> --crs_iface=<interface>
```
where `<sn>` is replaced with the CRS serial number (e.g. 27) and `<interface>` is replaced with the interface (e.g. 'enp2s0'). This command will run a variety of tests on the hardware, including sweeps and streaming. It should take 4-5 min to run. It will ensure that streamed data matches sweep data. It will also produce an output plot of sweeps over the full range for each module and a test results log saved in `<repository root>/crs_tests/`. The test log contains timing info for each `sweep_span` and `capture_ts` call, and maximum absolute differences between sweep data and timestream data.

## Description of CRS instrument functions 
<ul>
    <li> <b> __init__: </b> Initializes some class attributes and checks that the interface exists.</li>
    <li> <b> configure_system: </b> Configures the board to prepare for operation with some default parameters. Needs to be separate from __init__ because it is asynchronous. </li>
    <li> <b> set_clock_source: </b> Sets the clock source to either internal 'VCXO' for external 'SMA'. </li>
    <li> <b> set_analog_bank_high: </b> Sets the active modules to either 1-4 or 5-8, and sets the full scale dBm output on each module (usually set to the max of 7 dBm). </li>
    <li> <b> set_extended_bw: </b> Allows the user to switch the per-module bandwidth between 500 and 600 MHz, tough extra loss will occur at the edges if 600 MHz is selected. </li>
    <li> <b> set_decimation: </b> Sets the decimation stage, which determines the sample frequency. Also has options to select either 128 or 1024 channels per module and the limit the number of modules that are streamed. </li>
    <li> <b> set_nco: </b> Sets NCO frequencies. </li>
    <li> <b> disable_modules: </b> Removes NCO frequencies and clears channels. </li>
    <li> <b> write_tones: </b> Write tone frequencies and amplitudes. </li>
</ul>
The sweep and streaming functions are described below.

## Board Configuration

In [ ]:
# Board initialization 
import numpy as np
import zarr
from citkid.crs.instrument import CRS

crs = CRS(
    serial_number = 45, 
    interface = 'enp5s0'
)

await crs.configure_system(
    clock_source = "VCXO",
    full_scale_dbm = 7,
    analog_bank_high = False,
    verbose = True
)

In [ ]:
# Set NCOs 
await crs.set_nco(
    {1: 0.5e9, 
     2: 1.0e9, 
     3: 1.5e9, 
     4: 2.0e9
    }
)

await crs.set_extended_bw(
    extended = False
) # True for 600 MHz bandwidth with loss at edges

# Sweeps 
<ul>
    <li> <b> sweep: </b> Most flexible sweep function. Takes a 2D array of frequencies, where the first axis is the tone index and the second is the list of sweep frequency values, for which S21 is measured sequentially. Tone powers are constant - if you want flexibility here ask LF to add a function. Outputs from all modules are cleared before and after sweeping (again, ask LF if you want more flexibility). <i>nsamps</i> is the number of samples per tone. To determine the amount of averaging time per tone, divide <i>nsamps</i> by the sample frequency, which is always set to be ~596 Hz for sweeps through <i>dec_stage = 6</i>. Note that the full sweep will take longer due to networking overhead. 
    </li>
    <li> <b> sweep_span: </b> Takes a 1D list of tone frequencies <i> fres </i> and a single span and sweeps each tone over the span. See docstring for details. Some options include: (center, span) vs (start, span), upward vs downward sweep, log vs linear spacing. 
    </li>
    <li> <b> sweep_qres: </b> Takes a 1D list of tone frequencies and a list of spans for each frequency, given as a q-like factor. Performs a linear sweep over each tone. </li>
    <li> <b> sweep_full: </b> Sweeps over the full bandwidth of each module's set NCO, and flattens the output. Can be spaced linearly or logarithmically. </li>
</ul>

In [ ]:
# Example of sweep_full 
f, z = await crs.sweep_full(
    amplitude = -50, 
    npoints_per_tone = 10, 
    nsamps = 100, 
    log = False,
    verbose = True
)

## Timestreams 
Streaming was split into two functions, so that the user has the option of taking multiple timestreams after writing tones. We have deliberately dropped the word "noise" in favor of "timestream" (or "ts") within all CRS code. 
<ul> 
    <li> <b> stream: </b> Streams data for currently written tones. Automatically optimizes <i> dec_stage </i> settings given the tones that are written. Raises an error if the current combination of tones and <i> dec_stage </i> will drop packets. For fast streaming, aim for < 128 channels per module and as few modules as possible. After streaming, this function processes the data in batches and saves to a Zarr file. This step is important for sorting channels into the order specified by <i> crs.ch_map </i>, and chunking the output in a way that is convenient. <i> batch_size_mb </i> specifies the amount of data, in MB, that will be read to memory at a time. <i> chunk_size_mb </i> specifies the chunk size, in MB, for the output Zarr file. Generally, <i> chunk_size_mb </i> should be kept small, because only integer multiples of it can be saved to disk, so the final chunk may be mostly empty. However, it should not be so small that chunking overhead starts to eat into read/write times. 128 MB is typically a safe value. 
    </li>
        <li> <b> capture_ts: </b> Writes tones, then calls <i> stream. </i> 
        </li>
</ul>

In [ ]:
# Example of capture_ts
fres = np.linspace(260e6, 2240e6, 1000)
ares = np.ones_like(fres) * -50 
ts_duration_s = 10 
dec_stage = 5 
tmp_directory = 'tmp/'
root = zarr.open(tmp_directory + 'test_zarr.py', mode = 'a')
grp = root.require_group('noise_0') 


await crs.capture_ts(
            fres,
            ares, 
            ts_duration_s,
            dec_stage,
            grp,
            ch_map = None,
            allow_missing = False,
            tmp_directory = 'tmp/',
            batch_size_mb = 1000,
            chunk_size_mb = 128,
            delete_parser_data = True,
            verbose = True
    )

### Streaming data format 
The streamed data is stored in the Zarr file as key 'z', which is an array of int32, where axis 0 is (real, imag), axis 1 is the tone index, and axis 2 is the timestream index. A 1D array of scaling factors 'counts_to_dbc' is saved separately (indexed by tone) to scale from int32 -> float64. If loading in chunks, index before operating on z to avoid loading the whole timestream. 

In [ ]:
# Reading timestream data 
root = zarr.open(tmp_directory + 'test_zarr.py', mode = 'r')
grp = root['noise_0'] 

# Get chunk size on axis 2
chunk_idx = grp['z'].chunks[2] 
# Load first chunk
z = np.array(grp['z'][:, :, :chunk_idx])
# Load scaling factor to convert to dbc
counts_to_dbc = np.array(grp['counts_to_dbc'])
z = z * counts_to_dbc[:, np.newaxis] # casts to float64
# Flatten axis 0 to complex128
z = z[0] + 1j * z[1]

# Everything below this point is a work in progress
## Procedures
### target_sweep

In [ ]:
root = zarr.open(tmp_directory + 'test_zarr.py', mode = 'a')
grp = root.require_group('noise_0') 